In [1]:
DATASET = "./identifier_base_dataset.csv"

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn import metrics
import joblib
import re

df = pd.read_csv(DATASET, on_bad_lines='skip')
df.dropna(subset=['type', 'raw'], inplace=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(df['raw'], df['type'], test_size=0.9, random_state=42)

# Custom transformer to filter out numerical strings
class NumericalFilter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [' '.join([word for word in doc.split() if not re.match(r'^\d+(\.\d+)?$', word)]) for doc in X]

# Custom tokenizer to remove numbers
def custom_tokenizer(text):
    tokens = text.split()
    filtered_tokens = [token for token in tokens if not re.match(r'^\d+$', token)]
    return filtered_tokens

# Create a pipeline that vectorizes the text data and filters numerical strings
pipeline = Pipeline([
    ('filter', NumericalFilter()),
    ('vectorizer', CountVectorizer(tokenizer=custom_tokenizer)),
    ('classifier', MultinomialNB())
])

# Train the model
pipeline.fit(X_train, y_train)

# Evaluate the model
y_pred = pipeline.predict(X_test)
print(metrics.classification_report(y_test, y_pred))

# Print the features
vectorizer = pipeline.named_steps['vectorizer']
try:
    text_features = vectorizer.get_feature_names_out()
except AttributeError:
    text_features = vectorizer.get_feature_names()
print("Text Features:\n", text_features)

# Save the model to disk
model_filename = 'log_type_classifier.pkl'
joblib.dump(pipeline, model_filename)

# Function to load the model
def load_model(model_filename):
    return joblib.load(model_filename)

# Function to predict the log type
def predict_log_type(raw_log, model):
    return model.predict([raw_log])[0]

# Function to retrain the model with new data
def retrain_model(new_data, new_labels, model):
    # Convert the new data and labels to DataFrame
    new_df = pd.DataFrame({'raw': new_data, 'type': new_labels})
    
    # Append the new data to the existing training data
    global X_train, y_train
    X_train = X_train.append(new_df['raw'], ignore_index=True)
    y_train = y_train.append(new_df['type'], ignore_index=True)
    
    # Retrain the model with the updated training data
    model.fit(X_train, y_train)
    
    # Save the updated model to disk
    joblib.dump(model, model_filename)
    return model


/Users/andrewcampagna/Library/Python/3.8/lib/python/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

     Android       1.00      1.00      1.00       951
      Apache       0.99      1.00      1.00      1810
         BGL       0.99      1.00      1.00      1491
        HDFS       1.00      1.00      1.00      1813
         HPC       0.68      0.99      0.80      1789
   HealthApp       1.00      0.51      0.67      1742
       Linux       0.99      1.00      1.00      1788
         Mac       1.00      0.99      1.00      1371
     OpenSSH       1.00      1.00      1.00      1796
   OpenStack       1.00      1.00      1.00      1391
   Proxifier       0.99      1.00      1.00       934
       Spark       1.00      1.00      1.00      1237
 Thunderbird       1.00      1.00      1.00      1233

    accuracy                           0.95     19346
   macro avg       0.97      0.96      0.96     19346
weighted avg       0.97      0.95      0.95     19346

Text Features:
 ['!' '!=' '"account' ... 'zhangyan' '{name' '|']


In [3]:
# Example usage
if __name__ == "__main__":
    # Load the model
    loaded_model = load_model(model_filename)
    
    # Predict log type
    sample_log = "<13>May 18 15:47:16 Andrews-MacBook-Air-2 com.apple.xpc.launchd[1] (com.apple.mdworker.shared.0B000000-0000-0000-0000-000000000000[38976]): Service exited due to SIGKILL | sent by mds[82]\n"
    predicted_type = predict_log_type(sample_log, loaded_model)
    print(f"The predicted log type is: {predicted_type}")
    
    # If the prediction is wrong, correct it
    correct_type = "Mac"
    if predicted_type != correct_type:
        print(f"Updating model with correct log type: {correct_type}")
        # Retrain the model with the correct log type
        loaded_model = retrain_model([sample_log], [correct_type], loaded_model)
        
        # Predict again with the updated model
        updated_predicted_type = predict_log_type(sample_log, loaded_model)
        print(f"The updated predicted log type is: {updated_predicted_type}")

The predicted log type is: Spark
Updating model with correct log type: Mac


AttributeError: 'Series' object has no attribute 'append'